# Color Classifier — Google Colab Training

Dataset: [Color dataset for color recognition (Kaggle)](https://www.kaggle.com/datasets/adikurniawan/color-dataset-for-color-recognition)

**Important:** The default Kaggle set has **10 colors** (no **pink**). A pink image will be misclassified unless you add a `pink/` folder under `training_dataset/`.

Run cells in order. Enable **Runtime → Change runtime type → GPU (T4)** before training.

In [ ]:
# ========== CELL 1: Install TensorFlow + dependencies + check GPU ==========
!pip install -q tensorflow scipy pillow numpy

import tensorflow as tf
import numpy as np
import json
import os
from pathlib import Path

print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    print("GPU:", gpus[0])
    # Mixed precision = faster training on GPU
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision: ON")
else:
    print("WARNING: No GPU. Enable Runtime → Change runtime type → T4 GPU")

In [ ]:
# ========== CELL 2: Upload & extract dataset ==========
import zipfile
from google.colab import files

DATA_ROOT = Path("/content/training_dataset")
DATA_ROOT.mkdir(parents=True, exist_ok=True)

# --- Option A: Upload a ZIP from your computer (recommended) ---
print("Upload your dataset ZIP (must contain color folders: black, blue, brown, ...)")
uploaded = files.upload()  # pick the zip from Kaggle download

zip_name = next(iter(uploaded.keys()))
print("Extracting:", zip_name)

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall("/content")

# Find folder that contains class subfolders (black, blue, ...)
def find_dataset_root(base="/content"):
    base = Path(base)
    expected = {"black", "blue", "red", "white", "yellow"}
    for path in base.rglob("*"):
        if path.is_dir() and expected.issubset({p.name.lower() for p in path.iterdir() if p.is_dir()}):
            return path
    # fallback: training_dataset
    for name in ["training_dataset", "dataset", "color-dataset-for-color-recognition", "data"]:
        p = base / name
        if p.is_dir():
            return p
    return base / "training_dataset"

DATA_ROOT = find_dataset_root()
print("Dataset root:", DATA_ROOT)

classes = sorted([d.name for d in DATA_ROOT.iterdir() if d.is_dir()])
print("Classes found (", len(classes), "):", classes)

for c in classes:
    n = len(list((DATA_ROOT / c).glob("*")))
    print(f"  {c}: {n} files")

if "pink" not in [x.lower() for x in classes]:
    print("\nNOTE: No 'pink' folder — add pink images to training_dataset/pink/ for pink detection.")

In [ ]:
# ========== CELL 3: Train model (tuned, GPU-fast, accurate) ==========
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (224, 224)
BATCH_SIZE = 64 if tf.config.list_physical_devices("GPU") else 32
SEED = 42
MODEL_PATH = "/content/color_classifier.h5"
INDICES_PATH = "/content/class_indices.json"

tf.keras.utils.set_random_seed(SEED)

# Strong but color-safe augmentation
train_gen = ImageDataGenerator(
    rescale=1.0 / 255.0,
    validation_split=0.2,
    rotation_range=20,
    width_shift_range=0.12,
    height_shift_range=0.12,
    shear_range=0.08,
    zoom_range=0.12,
    horizontal_flip=True,
    brightness_range=(0.75, 1.25),
    channel_shift_range=25.0,
    fill_mode="nearest",
)
val_gen = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2)

train_data = train_gen.flow_from_directory(
    str(DATA_ROOT),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=SEED,
)
val_data = val_gen.flow_from_directory(
    str(DATA_ROOT),
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=SEED,
)

num_classes = train_data.num_classes
class_indices = {int(v): k for k, v in train_data.class_indices.items()}
with open(INDICES_PATH, "w") as f:
    json.dump(class_indices, f, indent=2)
print("num_classes:", num_classes)
print("class_indices:", class_indices)

def build_model(num_classes, trainable_base=False):
    base = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3),
    )
    base.trainable = trainable_base

    inputs = layers.Input(shape=(224, 224, 3))
    x = base(inputs, training=trainable_base)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    # float32 softmax for mixed precision stability
    outputs = layers.Dense(num_classes, activation="softmax", dtype="float32")(x)
    model = models.Model(inputs, outputs)
    return model

def compile_model(model, lr):
    model.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=["accuracy"],
    )

cb = [
    callbacks.ModelCheckpoint(
        MODEL_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1,
    ),
    callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=8,
        restore_best_weights=True,
        verbose=1,
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.35,
        patience=3,
        min_lr=1e-7,
        verbose=1,
    ),
]

print("\n--- Phase 1: frozen backbone (fast head training) ---")
model = build_model(num_classes, trainable_base=False)
compile_model(model, lr=3e-4)
model.summary()

history1 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=25,
    callbacks=cb,
)

print("\n--- Phase 2: fine-tune top MobileNetV2 layers ---")
base = model.layers[1]  # MobileNetV2
base.trainable = True
for layer in base.layers[:-50]:
    layer.trainable = False

compile_model(model, lr=1e-5)

history2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=30,
    callbacks=cb,
)

model.save(MODEL_PATH)
print("\nSaved:", MODEL_PATH)
print("Saved:", INDICES_PATH)

# Quick validation report
val_data.reset()
loss, acc = model.evaluate(val_data, verbose=0)
print(f"Final val_accuracy: {acc:.4f}  val_loss: {loss:.4f}")

print("\nDownload these files and replace in your Color Classifier folder:")
print("  - color_classifier.h5")
print("  - class_indices.json")
print("Then restart start_model_server.bat")

In [ ]:
# ========== CELL 4 (optional): Download trained model to your PC ==========
from google.colab import files
files.download("/content/color_classifier.h5")
files.download("/content/class_indices.json")